# Hakam — training

Trains the foul classifier and runs the two experiments the brief asks for.

Before you start: **Runtime → Change runtime type → L4 GPU**.

Then **Runtime → Run all** and leave it for about an hour.

The dataset is downloaded into this runtime and disappears with it. Only
metrics and model weights go to Drive — no video, per the NDA.

## 1. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
print(torch.cuda.get_device_name(0))

## 2. Get the code

In [ ]:
%cd /content
!git clone https://github.com/FerasMad/hakam.git 2>/dev/null || (cd hakam && git pull -q)
%cd /content/hakam
!pip install -q transformers SoccerNet opencv-python-headless pyzipper

## 3. Download the dataset

About 3.3 GB, four minutes. The password comes from the Colab secret
`SOCCERNET_PASSWORD` — set it with the key icon in the left sidebar.

It has to be passed as an *argument*; setting `.password` alone gives HTTP 401.

In [ ]:
import os
from pathlib import Path

from SoccerNet.Downloader import SoccerNetDownloader

try:
    from google.colab import userdata

    pw = userdata.get("SOCCERNET_PASSWORD")
except Exception:
    from getpass import getpass

    pw = getpass("Secret not found - enter the SoccerNet password: ")

os.environ["SOCCERNET_PASSWORD"] = pw

root = Path("/content/hakam/data/mvfouls")
downloader = SoccerNetDownloader(LocalDirectory=str(root))
downloader.password = pw
downloader.downloadDataTask(
    task="mvfouls", split=["train", "valid", "test"], password=pw
)

size_gb = sum(p.stat().st_size for p in root.rglob("*") if p.is_file()) / 1e9
print(f"
{size_gb:.2f} GB downloaded")
assert size_gb > 0.5, "Download failed. An HTTP 401 means the password is wrong."

## 4. Unpack the splits

The archives are AES-encrypted, so this needs the same password as the
download — it comes through the environment rather than a flag, to keep it
out of the notebook output.

The downloader also nests everything a level deeper than requested, and each
zip contains `action_0`, `action_1`, … so the splits would overwrite each
other if unpacked together.

You should see **2916 / 411 / 301** actions.

In [ ]:
!SOCCERNET_PASSWORD="$SOCCERNET_PASSWORD" python scripts/normalise_colab_layout.py

## 5. Cache the frames

Each incident has 2–4 camera angles. We keep only the **last** one, which is a
close-up in 95% of cases and played at about half speed — the clearest look at
the contact, and a third of the data to process.

Frames are stored at their original 224×398 so that step 8 can compare cropping
against resizing without decoding everything twice. Takes about four minutes.

In [ ]:
!python scripts/cache_frames.py --splits train valid test

## 6. Baseline — frozen backbone

Keep the pretrained model fixed and fit a simple classifier on top. Expect
roughly **0.55–0.58** balanced accuracy: weak on purpose. It is the comparison
point that shows how much fine-tuning actually buys.

In [ ]:
!python -m src.models.train --mode probe --stage card --geometry crop \
    --batch-size 32 --num-workers 2 --name card_probe_crop

## 7. Experiment 1 — fine-tuning

Same data, same everything, but now the model itself learns. This is the run
that should move the number. About 35 minutes.

In [ ]:
!python -m src.models.train --mode finetune --stage card --geometry crop \
    --augment mild_aug_v1 --freeze-blocks 6 --epochs 8 \
    --batch-size 8 --num-workers 2 --save --name card_finetune_crop

## 8. Experiment 2 — show the model the whole frame

The standard preprocessing crops a 224-wide square out of a 398-wide frame,
throwing away 87 pixels from each side. We measured 57 clips: in **12% of them
the foul itself lands outside that crop**, so the model is judging a clip that
no longer contains the incident.

This run keeps the full frame instead. Everything else is identical. About 35
minutes.

In [ ]:
!python -m src.models.train --mode finetune --stage card --geometry resize \
    --augment mild_aug_v1 --freeze-blocks 6 --epochs 8 \
    --batch-size 8 --num-workers 2 --save --name card_finetune_resize

## 9. Results

`review_load` is the share of incidents a human would have to check. Recall
always appears next to it — on its own it means nothing, since flagging every
clip scores perfect recall.

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
for path in sorted(Path("artifacts/runs").glob("*/metrics.json")):
    m = json.loads(path.read_text())
    tuned = next(v for k, v in m.items() if k.startswith("at_recall"))
    rows.append({
        "run": m["run"],
        "balanced_acc": m["argmax"]["balanced_accuracy"],
        "recall": tuned["recall_card"],
        "review_load": tuned["review_load"],
        "confident_acc": m["selective"]["selective_accuracy"],
        "minutes": m["minutes"],
    })

print(pd.DataFrame(rows).to_string(index=False))

## 10. Save results to Drive

Sign in with **hakamtuw@gmail.com** when prompted. Metrics and weights only.

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

destination = Path("/content/drive/MyDrive/hakam/runs")
destination.mkdir(parents=True, exist_ok=True)

for run in sorted(p for p in Path("artifacts/runs").glob("*") if p.is_dir()):
    shutil.copytree(run, destination / run.name, dirs_exist_ok=True)
    print(run.name)

print(f"\nsaved to {destination}")